<a href="https://colab.research.google.com/github/martinpius/ARCHITECTURES/blob/main/ST_7203_%3EIntro_to_Big_Data_Processing_with_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("ST7203").getOrCreate()

In [ ]:
spark

In [ ]:
import re
from google.colab import output
ui = spark.sparkContext.uiWebUrl

In [ ]:
ui

In [ ]:
a = re.search(r":(\d+)", ui)

In [ ]:
a

In [ ]:
port = int(re.search(r":(\d+)", ui).group(1))


In [ ]:
port

In [ ]:
output.serve_kernel_port_as_window(port = port, path = "/jobs/")

In [ ]:
sc = spark.sparkContext

In [ ]:
rdd = sc.parallelize(list(range(1, 10, 2)))

In [ ]:
rdd.take(1)

In [ ]:
text = ["Big Data Hadoop Map Reduce",
        "Hadoop Map Reduce",
        "Big Data with Spark",
        "Spark for Big Data",
        "Python for Big Data"]

In [ ]:
def mapper(t: str)-> list:
    return [w.upper() for sn in t for w in sn.split(" ")]


In [ ]:
words = mapper(t = text)

In [ ]:
words

In [ ]:
def shuffle(t: str) -> list:
    return [(w, 1) for w in words]

In [ ]:
words_c = shuffle(t = words)

In [ ]:
words_c

In [ ]:
from collections import defaultdict

In [ ]:
def reducer(t: str) -> tuple:
    dict1 = defaultdict(list)
    for tp in t:
        dict1[tp[0]].append(tp[1])
     # reducing
    out = {}
    for k, v in dict1.items():
        out[k] = sum(v)

    return dict1, out


In [ ]:
dfd, wcs = reducer(t = words_c)

In [ ]:
dfd, wcs

In [ ]:
rdd_text = sc.parallelize(text)

In [ ]:
rdd_ws = rdd_text.flatMap(lambda x: x.split(" "))


In [ ]:
rdd_ws.collect()

In [ ]:
rdd_wc = rdd_ws.map(lambda x: (x.upper(), 1))
rdd_wc.take(3)

In [ ]:
wc = rdd_wc.reduceByKey(lambda a, b: a + b)

In [ ]:
wc.collect()

In [ ]:
rdd = sc.parallelize(list(range(1, 100, 1)))

In [ ]:
rdd_e = rdd.filter(lambda x: x % 20 == 0)
rdd_e.collect()

In [ ]:
rdd_t = rdd.filter(lambda x: ((x ** 2 - 5) % 2 == 0 and  x < 10))

In [ ]:
rdd_t.collect()

In [ ]:
from google.colab import drive
drive.mount("/content/drive/", force_remount = True)

In [ ]:
path_temp = "/content/drive/MyDrive/AGEN"

In [ ]:
import requests
import zipfile
import io
import os

# GTFS (General Transit Feed Specification): Bus feed (LA Metro)

url = "https://gitlab.com/LACMTA/gtfs_bus/-/raw/master/gtfs_bus.zip"

# Download
response = requests.get(url)

# Check status-->
print("Status code:", response.status_code)

# Save and unzip if successful
if response.status_code == 200: #Files downloaded
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall("gtfs_data")
    print("Download and extraction complete.")
else:
    print("Download failed.")

In [ ]:
os.listdir("gtfs_data")[:2]

In [ ]:
!cp -r gtfs_data /content/drive/MyDrive/GTFSDATA

In [ ]:
path = "/content/drive/MyDrive/GTFSDATA/gtfs_data/stop_times.txt"

In [ ]:
# Read/Load an rdd from the given path using sparkContext
text_rdd = sc.textFile(path)

In [ ]:
text_rdd.take(2)

In [ ]:
# Filter out header
header = text_rdd.first()

In [ ]:
header

In [ ]:
# Filter the data
data_rdd = text_rdd.filter(lambda line: line != header)


In [ ]:
data_rdd.take(4)

In [ ]:
# Split by comma
fields_rdd = data_rdd.map(lambda line: line.split(','))


In [ ]:
fields_rdd.take(2)

In [ ]:
header

In [ ]:
# Extract trip_id and departure_time
trip_time_rdd = fields_rdd.map(lambda fields: (fields[0], fields[2]))  #

In [ ]:
trip_time_rdd.take(4)

In [ ]:
# Count trips by ID
trip_counts = trip_time_rdd.map(lambda x: (x[0], 1)).\
reduceByKey(lambda a, b: a + b)


In [ ]:
trip_counts.take(4)

In [ ]:
# Find trip with most stops
top_trip = trip_counts.takeOrdered(100, key = lambda x: -x[1])

In [ ]:
top_trip

In [ ]:
# Count the number of unique trips
unique_trips_rdd = data_rdd.map(lambda line: line.split(',')[0]).distinct()
print("Total unique trips (RDD):", unique_trips_rdd.count())

In [ ]:
header

In [ ]:
# Filter late than 18:00hrs  arrivals
evening_trips_rdd = data_rdd.filter(lambda line: int(line.split(',')[1].split(":")[0]) >= 18)


In [ ]:
evening_trips_rdd.take(2)

In [ ]:
# Number of stops at each stop ID
stop_counts_rdd = data_rdd.\
map(lambda line: (line.split(',')[3], 1)).\
reduceByKey(lambda a,b: a+b)


In [ ]:
stop_counts_rdd.take(3)

In [ ]:
trip_stop_rdd = data_rdd.map(lambda line: (line.split(',')[0], 1)).reduceByKey(lambda a,b: a+b)

In [ ]:
trip_stop_rdd.take(2)

In [ ]:
# Average number of stops per trip
trip_stop_rdd = data_rdd.map(lambda line: (line.split(',')[0], 1)).reduceByKey(lambda a,b: a+b)
avg_stops = trip_stop_rdd.map(lambda x: x[1]).mean()


In [ ]:
avg_stops

In [ ]:
header

In [ ]:
trip_stops_rdd = data_rdd.\
map(lambda line: line.split(',')) \
    .map(lambda x: (x[0], (int(x[4]), x[3])))

In [ ]:
trip_stops_rdd.take(3)

In [ ]:
# groupByKey → find first and last stop_id
trip_grouped = trip_stops_rdd.groupByKey().\
mapValues(lambda stops: sorted(list(stops)))

In [ ]:
l = [1,2,3]
l[-1]

In [ ]:
# Trips that start and end at the same stop [RUN TO CHECK]
trip_stops_rdd = data_rdd.\
map(lambda line: line.split(',')) \
    .map(lambda x: (x[0], (int(x[4]), x[3])))  # (trip_id, (stop_sequence, stop_id))
print(f"\n>>>> trip_stop obs: \n{trip_stops_rdd.take(3)}\n")
# groupByKey → find first and last stop_id
trip_grouped = trip_stops_rdd.groupByKey().\
mapValues(lambda stops: sorted(list(stops)))
print(f"\n>>>> trip_grouped obs: \n{trip_grouped.take(3)}\n")
same_stop_trips = trip_grouped.filter(lambda x: x[1][0][1] == x[1][-1][1])
print(f"\n>>>> same stops obs: \n{same_stop_trips.take(3)}\n")

In [ ]:
# Spark Data Frame
#import requests, zipfile
from pyspark.sql.types import StructType, StringType, StructField, IntegerType, DoubleType, FloatType, DateType

In [ ]:
cols = StructType([StructField("name", StringType(), True),
                   StructField("age", IntegerType(), True)]) # Define schema

In [ ]:
df_rdd = sc.parallelize([["anna", 10], ["bakari", 19]])

In [ ]:
df_rdd.collect()

In [ ]:
df_names = spark.createDataFrame(df_rdd, schema = cols)

In [ ]:
df_names.show()